# PyTorch로 이해하는 딥러닝 학습 원리 — 6회 미니 과제

**회당 20~30분 · Python 기초 문법을 아는 학습자 · CPU 실행 · 외부 데이터 다운로드 없음**

목표는 모델을 크게 만드는 것이 아니라, “손실이 어떻게 가중치 수정으로 이어지는가”를 코드로 설명하는 것입니다.
앞에서 배운 `backward → step`을 XOR, 다중 분류, 깊은 신경망, 옵티마이저 비교로 확장합니다.

| 회차 | 새 개념 | 앞 회차와의 연결 | 직접 수정할 것 |
|---|---|---|---|
| 1 | 순전파, 손실, 역전파, SGD | 한 개의 가중치에서 학습 시작 | 학습률과 기울기 초기화 |
| 2 | MLP, 비선형 활성화, XOR | 1회의 학습 단계를 반복 | 은닉층 활성화 제거·복원 |
| 3 | BCEWithLogitsLoss, 로짓, 수치 안정성 | 2회의 이진 분류 손실 분석 | Sigmoid 중복·shape·dtype 오류 |
| 4 | CrossEntropyLoss, 다중 분류 | 이진 분류에서 3개 클래스 분류로 확장 | 출력 차원·손실·정답·예측 방식 |
| 5 | 연쇄법칙, 기울기 소실 | 1회의 역전파를 여러 층으로 확장 | 깊이와 활성화 변경 |
| 6 | SGD, Momentum, Adam, 평가 | 같은 MLP에 서로 다른 갱신 규칙 적용 | 공정한 비교·학습률 변경 |

**매회 진행:** 개념 읽기 5분 → 예제 실행 5분 → 코드 수정 10분 → 정답·결과 해설 5분.
문제를 먼저 풀고 정답 셀을 실행하세요. 모든 코드 셀은 완성된 예제 또는 정답이며,
미완성 문제 코드는 설명 셀에 있으므로 `Run All`도 가능합니다.
각 회차의 첫 예제는 앞 회차의 변수를 실행하지 않아도 돌아갑니다.
정답 셀은 별도 안내가 없다면 해당 회차 예제 실행 후 사용합니다.

## 실행 준비

Jupyter 또는 Google Colab에서 이 노트북을 여세요. PyTorch가 없으면 코드 셀을 하나 추가해
`%pip install torch`를 한 번 실행하고 필요하면 커널을 재시작하세요.
일반 터미널에서는 `python -m pip install torch`를 사용합니다.
GPU는 필요하지 않습니다. 실험의 작은 데이터에 맞춰 스레드를 하나로 제한했습니다.
각 예제는 기본 CPU, float32를 사용하고, 5회만 작은 기울기를 보기 위해 float64를 씁니다.
실제 검증 환경과 출력은 각 코드 셀 아래에 기록되어 있습니다.
PyTorch 버전·운영체제에 따라 난수 초기화와 마지막 소수점은 달라질 수 있습니다.

**기록 규칙:** 수정 전에 예상 결과 한 줄, 실행 후 관찰 한 줄, 그 이유 한 줄을 적으세요.


In [1]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("device: CPU")


Python: 3.12.14
PyTorch: 2.14.1+cpu
device: CPU


## 1회 — backward는 무엇을 바꾸는가?

**핵심 개념**

가장 작은 모델을 $\hat y=wx$, 손실을 $L=(\hat y-y)^2$로 둡니다.
이번에는 MLP의 한 연결에 해당하는 가중치 `w` 하나만 학습합니다.
`x=2, y=8, w=1`이면 예측은 2, 손실은 36입니다.

연쇄법칙으로 $\frac{dL}{dw}=2(wx-y)x=-24$입니다.
SGD는 $w \leftarrow w-\eta\frac{dL}{dw}$로 갱신합니다.
학습률 $\eta=0.1$이면 새 가중치는 $1-0.1(-24)=3.4$입니다.

`loss.backward()`는 기울기를 `.grad`에 누적합니다.
`optimizer.step()`이 가중치를 수정하고, `zero_grad()`가 이전 기울기를 지웁니다.
`zero_grad(set_to_none=True)`는 저장된 기울기를 `None`으로 되돌립니다.
다음 backward에서 새 기울기가 만들어집니다.

**실행 예제 — 한 번의 학습 단계를 관찰하세요.**


In [2]:
import torch

torch.set_num_threads(1)
x = torch.tensor(2.0)
y = torch.tensor(8.0)
w = torch.nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([w], lr=0.1)

optimizer.zero_grad(set_to_none=True)
prediction = w * x
loss = (prediction - y) ** 2
print(f"before: w={w.item():.4f}, loss={loss.item():.4f}")

loss.backward()
print(f"after backward: w={w.item():.4f}, grad={w.grad.item():.4f}")
optimizer.step()

with torch.no_grad():
    new_loss = (w * x - y) ** 2
print(f"after step: w={w.item():.4f}, loss={new_loss.item():.4f}")


before: w=1.0000, loss=36.0000
after backward: w=1.0000, grad=-24.0000
after step: w=3.4000, loss=1.4400


**코드 수정 문제**

1. 예제를 처음부터 다시 실행하되 `lr=0.05`로 바꾸세요. 새 `w`와 손실을 실행 전에 계산하세요.
2. `w=1`로 다시 시작하고, **step 없이** 새 순전파와 backward를 두 번 반복하세요.
   반복마다 `zero_grad()`를 호출할 때와 생략할 때 기울기가 어떻게 다른지 출력하세요.
3. 아래 빈칸 세 곳에 `zero_grad(set_to_none=True)`, `backward()`, `step()`을 올바르게 배치하세요.

```python
optimizer.________
loss = (w * x - y) ** 2
loss.________
optimizer.________
```

**정답 — 학습률 수정과 누적 실험**


In [3]:
import torch

w = torch.nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([w], lr=0.05)
optimizer.zero_grad(set_to_none=True)
loss = (w * 2 - 8) ** 2
loss.backward()
optimizer.step()
with torch.no_grad():
    print(f"lr=0.05: w={w.item():.4f}, loss={((w * 2 - 8) ** 2).item():.4f}")

for reset in [True, False]:
    w = torch.nn.Parameter(torch.tensor(1.0))
    optimizer = torch.optim.SGD([w], lr=0.1)
    gradients = []
    for _ in range(2):
        if reset:
            optimizer.zero_grad(set_to_none=True)
        loss = (w * 2 - 8) ** 2  # 매번 새 그래프 생성
        loss.backward()
        gradients.append(w.grad.item())
        # 이 실험에서는 가중치를 바꾸지 않음
    print(f"reset={reset}: gradients={gradients}")


lr=0.05: w=2.2000, loss=12.9600
reset=True: gradients=[-24.0, -24.0]
reset=False: gradients=[-24.0, -48.0]


**실행 결과 해설**

첫 예제에서 backward 직후에도 `w=1`입니다. 기울기 계산과 파라미터 갱신은 다른 일입니다.
step 뒤에는 `w=3.4`, 예측은 6.8, 손실은 1.44가 됩니다.
학습률을 0.05로 줄이면 `w=2.2`, 손실은 12.96입니다. 이 한 번의 갱신에서는 이동량이 작아졌습니다.
“작은 학습률이면 항상 더 좋은 결과”라는 뜻은 아닙니다.

초기화한 기울기는 `[-24, -24]`, 누적한 기울기는 `[-24, -48]`입니다.
두 번째 실험은 step을 생략했기 때문에 계산되는 미분 자체가 매번 같습니다.
같은 `loss`에 무조건 backward를 두 번 호출하면 그래프 재사용 오류가 날 수 있으므로,
여기서는 매번 순전파를 다시 계산했습니다. 일반 학습 루프도 같은 방식입니다.

**완료 기준:** backward와 step의 차이, zero_grad가 필요한 이유를 각각 한 문장으로 설명하세요.


## 2회 — MLP로 XOR 학습하기

**핵심 개념**

MLP(Multi-Layer Perceptron)는 `Linear → 활성화 → Linear`처럼 완전연결층을 쌓은 모델입니다.
이번 모델의 식은 $h=\sigma(W_1x+b_1)$, $z=W_2h+b_2$입니다.
은닉층의 Sigmoid가 비선형성을 만듭니다. 출력 `z`는 확률로 바꾸기 전 점수인 **로짓**입니다.
가중치는 `nn.Linear`가 만들고, 1회에서 수행한 기울기 계산과 갱신을 반복합니다.

XOR 정답은 `[0, 1, 1, 0]`이며 한 개의 직선으로 두 클래스를 분리할 수 없습니다.
활성화 없이 Linear만 여러 개 쌓으면 하나의 아핀 변환으로 합쳐지므로 이 한계가 유지됩니다.

학습 손실에는 원래 로짓을 전달합니다. 확률을 확인할 때만 `sigmoid(logits)`를 사용합니다.
이 이유는 3회에서 직접 확인합니다. Adam은 우선 갱신 도구로 사용하고 6회에서 SGD와 비교합니다.


In [4]:
import torch
from torch import nn

torch.set_num_threads(1)
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

def train_xor(use_activation=True):
    torch.manual_seed(7)
    model = nn.Sequential(
        nn.Linear(2, 8),
        nn.Sigmoid() if use_activation else nn.Identity(),
        nn.Linear(8, 1),
    )
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
    with torch.no_grad():
        initial_loss = criterion(model(X), y).item()
    model.train()
    for _ in range(2000):
        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probabilities = logits.sigmoid()
        predictions = (logits >= 0).long()
        final_loss = criterion(logits, y).item()
        accuracy = (predictions == y.long()).float().mean().item()
    print(f"activation={use_activation}: loss {initial_loss:.4f} -> {final_loss:.4f}")
    print("probabilities:", [round(v, 4) for v in probabilities.flatten().tolist()])
    print("predictions:", predictions.flatten().tolist(), f"accuracy={accuracy:.0%}")
    return model

model = train_xor(True)


activation=True: loss 0.7226 -> 0.0000
probabilities: [0.0, 1.0, 1.0, 0.0001]
predictions: [0, 1, 1, 0] accuracy=100%


**코드 수정 문제**

1. `train_xor(False)`로 실행해 은닉층의 Sigmoid를 제거하세요. epoch 수가 같아도 XOR를 학습할 수 있나요?
2. 활성화를 복원하고 은닉 차원을 8에서 16으로 바꾸세요. 두 Linear의 연결 차원도 함께 수정하세요.
3. 로짓 기준 예측을 확률 기준 예측으로 바꾸세요. 기준 확률은 얼마인가요?

**정답**

2번의 모델은 `nn.Linear(2, 16) → nn.Sigmoid() → nn.Linear(16, 1)`입니다.
중간 두 차원 16이 같아야 행렬 곱이 가능합니다. 차원을 바꾼 결과는 새 초기화에 따라 달라질 수 있습니다.
아래는 1번 비교와 3번 예측 방식입니다. 예제의 `train_xor`, `X`를 사용합니다.


In [5]:
linear_model = train_xor(False)
with torch.no_grad():
    logits = model(X)  # 활성화가 있는 예제 모델
    by_logit = (logits >= 0).long()
    by_probability = (logits.sigmoid() >= 0.5).long()
print("same predictions:", torch.equal(by_logit, by_probability))

# 2번 정답도 실제로 연결되는지 확인
torch.manual_seed(7)
wider_model = nn.Sequential(nn.Linear(2, 16), nn.Sigmoid(), nn.Linear(16, 1))
print("wider output shape:", tuple(wider_model(X).shape))


activation=False: loss 0.7039 -> 0.6931
probabilities: [0.5, 0.5, 0.5, 0.5]
predictions: [1, 1, 1, 1] accuracy=50%
same predictions: True
wider output shape: (4, 1)


**실행 결과 해설**

활성화가 있는 모델은 네 입력의 정답 `[0, 1, 1, 0]`을 학습합니다.
활성화가 없는 모델의 손실은 약 0.6931에 머뭅니다. 이는 균형 잡힌 이 XOR 데이터에서
모든 입력에 확률 0.5를 출력할 때의 평균 BCE인 $-\log(0.5)$와 같습니다.
0.5 근처의 아주 작은 오차 때문에 출력 클래스와 정확도는 흔들릴 수 있습니다.
핵심은 모든 XOR 입력을 올바르게 분리하지 못한다는 점입니다.

Sigmoid가 단조 증가하고 `sigmoid(0)=0.5`이므로 두 예측 방식은 같습니다.
이 정확도는 **네 학습 샘플의 재현 성공**을 뜻합니다. 새로운 데이터에 대한 일반화 성능은 아닙니다.

**완료 기준:** Linear만 쌓을 때 XOR를 해결할 수 없는 이유를 설명하세요.


## 3회 — BCEWithLogitsLoss와 Sigmoid 중복 오류

**핵심 개념**

이진 분류에서 $p=\sigma(z)$, $\ell=-[y\log p+(1-y)\log(1-p)]$입니다.
`BCEWithLogitsLoss`는 이 계산을 로짓 `z`에서 수치적으로 안정되게 수행합니다.
반면 `BCELoss`는 확률을 입력받습니다. 두 함수의 입력을 구분해야 합니다.

| 손실함수 | 입력 | 이번 과제의 정답 형태 |
|---|---|---|
| BCEWithLogitsLoss | 원래 로짓, 예: `(N, 1)` | 같은 shape의 float, 0 또는 1 |
| BCELoss | Sigmoid 적용 후 확률 | 같은 shape의 float, 0 또는 1 |

`BCEWithLogitsLoss(sigmoid(z), y)`도 숫자는 계산되지만 다른 목적함수입니다.
입력 확률을 다시 로짓으로 해석하므로 추가 Sigmoid 효과가 생깁니다.
평균 손실의 올바른 로짓 기울기는 $(\sigma(z)-y)/N$입니다. 여기서는 원소가 N개입니다.


In [6]:
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
z = torch.tensor([-2., 0., 2.], requires_grad=True)
y = torch.tensor([0., 1., 1.])
loss = nn.BCEWithLogitsLoss()(z, y)
manual = -(y * F.logsigmoid(z) + (1 - y) * F.logsigmoid(-z)).mean()
gradient = torch.autograd.grad(loss, z)[0]
expected_gradient = (z.detach().sigmoid() - y) / y.numel()
print(f"built-in={loss.item():.6f}, stable manual={manual.item():.6f}")
print("gradients:", [round(v, 6) for v in gradient.tolist()])
print("gradient formula matches:", torch.allclose(gradient, expected_gradient))

bad_z = torch.tensor([-2., 0., 2.], requires_grad=True)
bad_loss = nn.BCEWithLogitsLoss()(bad_z.sigmoid(), y)
bad_loss.backward()
print(f"double sigmoid loss={bad_loss.item():.6f}")
print("wrong-objective gradients:", [round(v, 6) for v in bad_z.grad.tolist()])

# 확신을 갖고 오답을 낸 극단값: 손실은 각각 약 1000이어야 함
extreme_z = torch.tensor([-1000., 1000.])
extreme_y = torch.tensor([1., 0.])
print("extreme stable loss:", F.binary_cross_entropy_with_logits(extreme_z, extreme_y).item())


built-in=0.315668, stable manual=0.315668
gradients: [0.039734, -0.166667, -0.039734]
gradient formula matches: True
double sigmoid loss=0.525114
wrong-objective gradients: [0.018541, -0.031462, -0.010255]
extreme stable loss: 1000.0


**코드 수정 문제**

다음 코드는 로짓에 Sigmoid를 먼저 적용하고, 정답 shape와 dtype도 맞지 않습니다.
세 부분을 고쳐 위 예제와 같은 손실을 얻으세요. `logits`는 모델의 마지막 Linear 출력이라고 가정합니다.

```python
logits = torch.tensor([[-2.], [0.], [2.]], requires_grad=True)  # (3, 1)
labels = torch.tensor([0, 1, 1])                             # (3,), long
loss = nn.BCEWithLogitsLoss()(logits.sigmoid(), labels)
```

추가 질문: `sigmoid(-1000)`이 0으로 반올림된 뒤 직접 `log(0)`을 계산하면 왜 위험할까요?

**정답**


In [7]:
import torch
from torch import nn

logits = torch.tensor([[-2.], [0.], [2.]], requires_grad=True)
labels = torch.tensor([0, 1, 1])
targets = labels.to(dtype=logits.dtype).view_as(logits)
loss = nn.BCEWithLogitsLoss()(logits, targets)
loss.backward()
print("shape:", tuple(logits.shape), tuple(targets.shape), "dtype:", targets.dtype)
print(f"corrected loss={loss.item():.6f}")
print("gradients:", [round(v, 6) for v in logits.grad.flatten().tolist()])


shape: (3, 1) (3, 1) dtype: torch.float32
corrected loss=0.315668
gradients: [0.039734, -0.166667, -0.039734]


**실행 결과 해설**

올바른 손실은 약 0.315668이고 기울기는 `[0.039734, -0.166667, -0.039734]`입니다.


정답 1인 마지막 원소는 기울기가 음수이므로 SGD로 갱신하면 로짓이 커지는 방향으로 움직입니다.
이중 Sigmoid는 추가 미분 인자를 만들며 학습하려는 손실도 바꿉니다.
고친 코드에서는 `logits`와 정답이 모두 `(3, 1)`, 정답은 float32입니다.

직접 `log(sigmoid(z))`를 계산하면 반올림된 0에 로그를 적용해 `-inf`가 될 수 있습니다.
내장 손실은 안정적인 식을 사용하므로, 극단값 예제에서도 약 1000인 유한한 손실을 냅니다.
`F.logsigmoid`를 이용한 수동 계산 역시 안정적인 비교용 계산입니다.

**완료 기준:** BCEWithLogitsLoss 앞에서 Sigmoid를 제거해야 하는 이유를 설명하세요.


## 4회 — CrossEntropyLoss로 3개 클래스 분류하기

**핵심 개념**

하나의 샘플이 여러 클래스 중 **정확히 하나**에 속하면 다중 클래스 분류입니다.
클래스가 C개이면 모델은 샘플당 C개의 로짓을 출력합니다.
클래스 인덱스 정답을 쓰는 이 과제에서 `CrossEntropyLoss`는 `(N, C)` 로짓과
`(N,)` 형태의 `torch.long` 정답을 받습니다. 정답 값은 `0`부터 `C-1`까지입니다.
이 함수는 확률 분포 형태의 정답도 지원하지만, 이번 과제에서는 사용하지 않습니다.

$$L=-\frac{1}{N}\sum_i\log\frac{\exp(z_{i,y_i})}{\sum_c\exp(z_{i,c})}$$

Softmax와 로그 손실을 합친 안정적인 계산을 하므로, 모델 끝에 Softmax를 넣지 않습니다.
예측 클래스는 `logits.argmax(dim=1)`로 고릅니다. 확률 표시가 필요할 때만 Softmax를 씁니다.
샘플 하나에 여러 라벨이 동시에 참인 **다중 라벨 분류**는 다른 설정이며 BCE 계열을 사용합니다.

**실행 예제 — 먼저 정해진 로짓으로 손실을 검산합니다.**


In [8]:
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
logits = torch.tensor([[2., 1., 0.], [0., 2., 1.]], requires_grad=True)
labels = torch.tensor([0, 1], dtype=torch.long)
loss = nn.CrossEntropyLoss()(logits, labels)
manual = -F.log_softmax(logits, dim=1)[torch.arange(len(labels)), labels].mean()
loss.backward()
print(f"built-in={loss.item():.6f}, manual={manual.item():.6f}")
print("predictions:", logits.detach().argmax(dim=1).tolist())
print("gradient row sums:", [round(v, 6) for v in logits.grad.sum(dim=1).tolist()])
wrong_loss = nn.CrossEntropyLoss()(logits.detach().softmax(dim=1), labels)
print(f"softmax before CE (wrong objective)={wrong_loss.item():.6f}")


built-in=0.407606, manual=0.407606
predictions: [0, 1]
gradient row sums: [0.0, -0.0]
softmax before CE (wrong objective)=0.797190


**코드 수정 문제 — 2회의 MLP를 3개 클래스 분류기로 바꾸세요.**

아래 데이터 생성 코드를 사용합니다. 외부 데이터 없이 서로 떨어진 3개 점 무리를 만듭니다.

```python
torch.manual_seed(11)
centers = torch.tensor([[-2., -1.], [2., -1.], [0., 2.]])
X = torch.cat([c + 0.25 * torch.randn(30, 2) for c in centers])
y = torch.arange(3).repeat_interleave(30)
```

2회의 구조를 `Linear(2, 8) → ReLU → Linear(8, ?)`로 변경하세요.
손실함수, 정답 shape·dtype, 예측 코드를 함께 바꾸세요.
Adam의 `lr=0.03`, 학습 반복은 150번으로 두세요. 학습 전후 손실과 정확도를 출력하세요.

**정답 — 네 수정 지점은 출력 3개, CrossEntropyLoss, `(N,)` long 정답, argmax입니다.**


In [9]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(11)
centers = torch.tensor([[-2., -1.], [2., -1.], [0., 2.]])
X = torch.cat([c + 0.25 * torch.randn(30, 2) for c in centers])
y = torch.arange(3).repeat_interleave(30)  # (90,), torch.long
model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 3))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
with torch.no_grad():
    initial_loss = criterion(model(X), y).item()

model.train()
for _ in range(150):
    optimizer.zero_grad(set_to_none=True)
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    logits = model(X)
    final_loss = criterion(logits, y).item()
    accuracy = (logits.argmax(dim=1) == y).float().mean().item()
print("logits / targets:", tuple(logits.shape), tuple(y.shape), y.dtype)
print(f"loss {initial_loss:.6f} -> {final_loss:.6f}, train accuracy={accuracy:.0%}")


logits / targets: (90, 3) (90,) torch.int64
loss 1.297287 -> 0.000135, train accuracy=100%


**실행 결과 해설**

고정 로짓의 손실은 약 0.407606이고 예측은 `[0, 1]`입니다.
Softmax를 먼저 적용하면 다른 손실이 출력됩니다. CE는 그 확률을 다시 로짓으로 해석합니다.
기울기 행의 합이 거의 0인 것은 각 샘플에서 CE 기울기가 `(softmax - one_hot) / N`이기 때문입니다.
부동소수점 계산에서는 아주 작은 잔차가 남을 수 있습니다.

MLP의 출력은 `(90, 3)`, 정답은 `(90,)`입니다. 이 예제는 잘 분리된 학습 데이터이므로
손실이 크게 줄고 학습 정확도는 100%에 도달합니다. 일반화를 판단할 검증 세트는 없습니다.
6회에서는 학습에 쓰지 않은 입력을 따로 평가합니다.

**완료 기준:** 이진 분류와 3개 클래스 분류의 출력·정답·손실함수 차이를 적으세요.


## 5회 — 깊어질수록 기울기가 작아지는 이유

**핵심 개념**

역전파는 각 층의 미분을 연쇄법칙으로 곱해 앞쪽으로 전달합니다.
Sigmoid의 도함수는 $\sigma(z)(1-\sigma(z))\leq 1/4$입니다.
작은 인자가 반복해서 곱해지면 앞쪽 가중치의 기울기가 매우 작아질 수 있습니다.
ReLU는 입력이 양수이면 도함수가 1, 음수이면 0입니다.

여기서는 현상을 분리해서 보기 위해 **폭 1, 모든 가중치 1, 편향 0**인 작은 깊은 MLP를 만듭니다.
`out.sum().backward()`는 출력의 입력·가중치에 대한 민감도를 측정합니다.
분류 손실로 학습하는 실험은 아니므로 step은 없습니다.
마지막 층과 첫 층의 가중치 기울기를 비교하고, 입력 기울기도 출력합니다.
실제 MLP에서는 가중치 행렬과 경로 합도 영향을 주므로, Sigmoid의 1/4만으로 전체 기울기를 단정할 수 없습니다.


In [10]:
import torch
from torch import nn

torch.set_num_threads(1)

def probe(activation="sigmoid", depth=8, input_value=1.0):
    modules = []
    linears = []
    for _ in range(depth):
        layer = nn.Linear(1, 1, bias=False, dtype=torch.float64)
        nn.init.ones_(layer.weight)
        linears.append(layer)
        modules += [layer, nn.Sigmoid() if activation == "sigmoid" else nn.ReLU()]
    model = nn.Sequential(*modules)
    x = torch.tensor([[input_value]], dtype=torch.float64, requires_grad=True)
    out = model(x)
    out.sum().backward()
    gradients = [layer.weight.grad.abs().item() for layer in linears]
    print(f"{activation}, depth={depth}, input={input_value:+.1f}")
    print("layer weight gradients:", [f"{g:.2e}" for g in gradients])
    print(f"input gradient={x.grad.item():.2e}")
    return gradients

sigmoid_gradients = probe("sigmoid", 8)
relu_gradients = probe("relu", 8)


sigmoid, depth=8, input=+1.0
layer weight gradients: ['5.52e-06', '2.05e-05', '8.63e-05', '3.79e-04', '1.68e-03', '7.48e-03', '3.33e-02', '1.48e-01']
input gradient=5.52e-06
relu, depth=8, input=+1.0
layer weight gradients: ['1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00']
input gradient=1.00e+00


**코드 수정 문제**

1. 깊이 8의 각 층 기울기를 보고, 첫 층과 마지막 층 중 어느 쪽이 작은지 기록하세요.
2. 깊이를 16으로 바꿔 두 활성화를 다시 비교하세요. Sigmoid의 첫 층 기울기는 더 작아지나요?
3. ReLU의 입력을 -1로 바꾸세요. “ReLU는 기울기 소실을 언제나 해결한다”는 주장이 맞나요?


**정답 — 예제의 `probe` 함수를 사용합니다.**


In [11]:
sigmoid_deep = probe("sigmoid", 16)
relu_deep = probe("relu", 16)
relu_negative = probe("relu", 8, input_value=-1.0)
print("sigmoid first gradient gets smaller:", sigmoid_deep[0] < sigmoid_gradients[0])
print("negative-input ReLU gradients all zero:", all(g == 0 for g in relu_negative))


sigmoid, depth=16, input=+1.0
layer weight gradients: ['3.58e-11', '1.33e-10', '5.61e-10', '2.46e-09', '1.09e-08', '4.86e-08', '2.16e-07', '9.63e-07', '4.28e-06', '1.91e-05', '8.48e-05', '3.78e-04', '1.68e-03', '7.48e-03', '3.33e-02', '1.48e-01']
input gradient=3.58e-11
relu, depth=16, input=+1.0
layer weight gradients: ['1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00', '1.00e+00']
input gradient=1.00e+00
relu, depth=8, input=-1.0
layer weight gradients: ['0.00e+00', '0.00e+00', '0.00e+00', '0.00e+00', '0.00e+00', '0.00e+00', '0.00e+00', '0.00e+00']
input gradient=0.00e+00
sigmoid first gradient gets smaller: True
negative-input ReLU gradients all zero: True


**실행 결과 해설**

Sigmoid는 앞쪽으로 갈수록 기울기가 작아지고 깊이 16에서는 현상이 더 심해집니다.
이 실험의 ReLU는 양수 입력과 단위 가중치 덕분에 모든 층의 기울기가 1로 유지됩니다.
하지만 음수 입력에서는 첫 ReLU가 경로를 차단해 모든 기울기가 0이 됩니다.
ReLU도 비활성 구간에서는 학습 신호를 전달하지 못한다는 예입니다.

해결책은 활성화 변경뿐 아니라 적절한 초기화, 정규화, 잔차 연결 등과 함께 검토해야 합니다.
학습률을 크게 하는 것만으로 손실된 전달 경로가 복원되지는 않습니다.
마지막 층 하나의 기울기만 보지 말고 여러 층을 비교하는 습관이 중요합니다.

**완료 기준:** 작은 기울기와 0인 기울기를 구분하고, ReLU 비교의 실험 조건을 설명하세요.


## 6회 — 같은 MLP에 SGD·Momentum·Adam 적용하기

**핵심 개념**

역전파가 계산한 기울기를 받아 파라미터를 갱신하는 규칙이 옵티마이저입니다.

| 옵티마이저 | 핵심 아이디어 |
|---|---|
| SGD | 현재 기울기의 반대 방향으로 이동 |
| SGD + Momentum | 이전 기울기의 이동 방향을 누적해 반영 |
| Adam | 기울기의 1차·2차 이동평균을 이용해 원소별 갱신 크기를 조절 |

1회의 회귀를 $y=x^2$로 확장하고 2회의 MLP, 5회의 ReLU를 결합합니다.
연속값 예측이므로 손실은 MSE입니다. 마지막 층은 출력 1개이고 Sigmoid는 붙이지 않습니다.
출력값이 0~1로 제한되어야 하는 분류 문제가 아니기 때문입니다.

비교하려면 데이터뿐 아니라 **초기 가중치와 갱신 횟수**도 같아야 합니다.
같은 `base_model`을 `deepcopy`해서 시작합니다.
모두 lr=0.03으로 비교하는 것은 고정 학습률 실험이며, 각 방법의 최적 성능 비교는 아닙니다.
실전 성능을 비교하려면 각 옵티마이저의 학습률을 따로 탐색해야 합니다.

학습 입력은 [-1, 1]의 41개 점, 검증 입력은 그 사이의 40개 중간점입니다.
검증 점은 역전파에 쓰지 않습니다. 같은 함수의 보간 실험이며 외삽이나 실제 데이터 성능을 보장하지 않습니다.
`model.eval()`은 평가 모드, `torch.no_grad()`는 기울기 기록 중지입니다. 두 기능은 서로 다릅니다.
이 모델에는 Dropout·BatchNorm이 없지만 평가 루틴을 연습하기 위해 둘 다 사용합니다.


In [12]:
import copy
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(7)
X_train = torch.linspace(-1, 1, 41).view(-1, 1)
y_train = X_train.square()
X_valid = torch.linspace(-0.975, 0.975, 40).view(-1, 1)
y_valid = X_valid.square()
base_model = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))

def compare_optimizers(lr=0.03, steps=200, include_momentum=False):
    factories = {
        "SGD": lambda params: torch.optim.SGD(params, lr=lr),
        "Adam": lambda params: torch.optim.Adam(params, lr=lr),
    }
    if include_momentum:
        factories["Momentum"] = lambda params: torch.optim.SGD(params, lr=lr, momentum=0.9)
    results = {}
    for name, factory in factories.items():
        model = copy.deepcopy(base_model)  # 같은 초기값으로 독립된 모델 생성
        optimizer = factory(model.parameters())
        criterion = nn.MSELoss()
        with torch.no_grad():
            initial = criterion(model(X_train), y_train).item()
        model.train()
        for _ in range(steps):
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X_train), y_train)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_loss = criterion(model(X_train), y_train).item()
            valid_loss = criterion(model(X_valid), y_valid).item()
        results[name] = (train_loss, valid_loss)
        print(f"{name:8s} lr={lr:g}, steps={steps}: initial={initial:.6f}, "
              f"train={train_loss:.6f}, valid={valid_loss:.6f}")
    return results

baseline = compare_optimizers()


SGD      lr=0.03, steps=200: initial=0.157737, train=0.077248, valid=0.069870
Adam     lr=0.03, steps=200: initial=0.157737, train=0.000896, valid=0.000779


**코드 수정 문제**

1. `torch.optim.SGD(..., momentum=0.9)`를 추가하고 세 옵티마이저를 비교하세요.
   예제에 준비된 `include_momentum` 분기를 사용해도 됩니다.
2. 세 방법 모두 학습률을 0.003으로 바꾸세요. 200번 갱신에서 검증 손실이 어떻게 달라지나요?
3. 왜 루프 안에서 새 모델을 만들거나 복제해야 하나요?
4. `loss.backward()`를 빼면 어떤 일이 생길지 설명하세요. 검증 손실이 계산되어도 학습이 일어났다고 볼 수 있나요?

**정답 — 같은 초기 상태에서 실험을 각각 다시 시작합니다.**


In [13]:
with_momentum = compare_optimizers(include_momentum=True)
smaller_lr = compare_optimizers(lr=0.003, include_momentum=True)
print("base model gradients untouched:", all(p.grad is None for p in base_model.parameters()))


SGD      lr=0.03, steps=200: initial=0.157737, train=0.077248, valid=0.069870
Adam     lr=0.03, steps=200: initial=0.157737, train=0.000896, valid=0.000779
Momentum lr=0.03, steps=200: initial=0.157737, train=0.003720, valid=0.003254
SGD      lr=0.003, steps=200: initial=0.157737, train=0.099415, valid=0.090385
Adam     lr=0.003, steps=200: initial=0.157737, train=0.027579, valid=0.024131
Momentum lr=0.003, steps=200: initial=0.157737, train=0.078763, valid=0.071265
base model gradients untouched: True


**실행 결과 해설**

세 방법의 초기 손실이 같아야 출발점이 통제된 것입니다.
실제 출력의 `train`, `valid`를 비교하고 학습률 변경 효과를 기록하세요.
이 구성에서는 Adam이 200번 갱신 안에 작은 손실에 도달하지만, 모든 문제에서 Adam이 이긴다는 뜻은 아닙니다.
학습률을 낮추면 200번 안에 충분히 이동하지 못할 수 있고, 더 많은 갱신이 필요할 수도 있습니다.
Momentum은 이전 방향을 활용하지만 진동하거나 지나칠 수도 있습니다.

같은 모델을 계속 사용하면 두 번째 방법이 첫 번째 방법의 학습 결과에서 시작하므로 비교가 섞입니다.
옵티마이저도 각 모델마다 새로 만들어 내부 상태를 초기화합니다.
deepcopy한 모델만 학습하므로 `base_model`에는 기울기가 저장되지 않습니다.

이 루프에서 backward를 지우면 zero_grad 뒤 모든 파라미터의 `.grad`가 None으로 남아
SGD·Adam이 해당 파라미터를 건너뜁니다. 손실 숫자는 출력할 수 있지만 가중치 학습은 일어나지 않습니다.
학습이 일어나는지 보려면 손실의 변화와 파라미터·기울기를 함께 확인하세요.

**완료 기준:** 초기값·데이터·갱신 횟수를 통제한 비교와, 최적 하이퍼파라미터 비교를 구분하세요.


## 마무리 복습 — 코드를 보지 않고 설명하기

1. `loss.backward()`와 `optimizer.step()`의 역할은 무엇인가요?
2. XOR에서 은닉층 활성화를 없애면 왜 표현력이 제한되나요?
3. BCEWithLogitsLoss와 CrossEntropyLoss의 입력에는 왜 확률 대신 로짓을 넣나요?
4. 이 과제에서 BCE 정답과 CE 정답의 shape·dtype은 어떻게 다른가요?
5. 마지막 층의 기울기는 큰데 첫 층은 아주 작다면 무엇을 살펴봐야 하나요?
6. Adam으로 바꾸기만 하면 기울기 소실이 반드시 해결되나요?

**간단 정답:** ① 미분 계산 / 가중치 갱신 ② 아핀 변환의 합성은 아핀 변환
③ 내장된 안정적인 확률·로그 손실 계산을 사용하기 위해
④ BCE: 로짓과 같은 shape의 float / 이번 CE: `(N,)` long 인덱스
⑤ 층별 기울기, 활성화 포화, 초기화와 깊이
⑥ 아니요. 옵티마이저가 바뀌어도 역전파 경로가 사라지거나 극도로 약해질 수 있습니다.

**학습 일지 양식**

| 변경한 코드 | 실행 전 예상 | 실제 출력 | 원인 해석 |
|---|---|---|---|
| 예: 2회 Sigmoid → Identity | XOR 학습 실패 | 손실 약 0.6931 | 비선형 표현력 상실 |

## 공식 참고 문서

설명과 API 사용법은 아래 PyTorch 공식 문서를 참고했습니다. 실험 데이터·문제·해설은 이 과제용으로 구성했습니다.

- [Autograd와 역전파](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)
- [BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)
- [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [torch.optim](https://docs.pytorch.org/docs/stable/optim.html)
